---
title: "Parallel Impact Investigation and Fan-In"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval, parallelism]
---

Change investigation is a map-reduce workflow: planning emits bounded evidence tasks, each branch owns its input, and synthesis waits for an accountable fan-in. `Send` expresses the dynamic branch count without sharing mutable branch state.


## Send maps evidence tasks into isolated state

The small graph below exposes the teaching-critical primitive directly. The reducer only appends branch results; completeness and duplicate checks belong in the join node.


In [1]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

class MapState(TypedDict):
    tasks: list[dict[str, int | str]]
    results: Annotated[list[dict[str, int | str]], operator.add]
    summary: dict[str, int]

class BranchState(TypedDict):
    task: dict[str, int | str]

def make_tasks(state: MapState):
    return {"tasks": [
        {"id": "code", "latency": 20},
        {"id": "tests", "latency": 30},
        {"id": "history", "latency": 40},
    ]}

def dispatch(state: MapState):
    return [Send("collect", {"task": task}) for task in state["tasks"]]

def collect(state: BranchState):
    task = state["task"]
    return {"results": [{"task_id": task["id"], "latency": task["latency"]}]}

def join(state: MapState):
    ids = [str(row["task_id"]) for row in state["results"]]
    assert len(ids) == len(set(ids)) == len(state["tasks"])
    return {"summary": {"branches": len(ids), "critical_path_ms": max(int(row["latency"]) for row in state["results"])} }

builder = StateGraph(MapState)
builder.add_node("make_tasks", make_tasks)
builder.add_node("collect", collect)
builder.add_node("join", join)
builder.add_edge(START, "make_tasks")
builder.add_conditional_edges("make_tasks", dispatch, ["collect"])
builder.add_edge("collect", "join")
builder.add_edge("join", END)
map_graph = builder.compile()
summary = map_graph.invoke({"tasks": [], "results": [], "summary": {}})["summary"]
print(summary)
assert summary == {"branches": 3, "critical_path_ms": 40}


{'branches': 3, 'critical_path_ms': 40}


The branches report ninety milliseconds of simulated work but a forty-millisecond critical path. The join, not the reducer, decides whether the evidence set is complete. This is the same contract used by the Change Planner's code, test, operations, and history branches.

## Parallelism changes cost and recovery


In [2]:
from change_planner.schemas import FaultPlan
from change_planner.workflow import run_fixture

parallel = run_fixture("dry-run-01", variant="full")
sequential = run_fixture("dry-run-01", variant="sequential")
gap = run_fixture("dry-run-01", faults=FaultPlan(missing_task_id="history"))
for name, state in (("parallel", parallel), ("sequential", sequential), ("missing branch", gap)):
    print(name, {
        "status": state["status"],
        "branches": len({row["task_id"] for row in state["branch_results"]}),
        "latency_ms": state["run_metrics"]["simulated_latency_ms"],
        "reason": state["terminal_reason"],
    })
assert parallel["run_metrics"]["simulated_latency_ms"] < sequential["run_metrics"]["simulated_latency_ms"]
assert gap["status"] == "failed" and "history" in gap["terminal_reason"]


parallel {'status': 'complete', 'branches': 4, 'latency_ms': 25, 'reason': 'completion contract satisfied'}
sequential {'status': 'complete', 'branches': 4, 'latency_ms': 85, 'reason': 'completion contract satisfied'}
missing branch {'status': 'failed', 'branches': 4, 'latency_ms': 25, 'reason': 'incomplete investigation branches: history'}


Parallelism reduces the critical path without changing the number of search operations. It also forces the workflow to account for every scheduled issue. A partial result is an explicit gap or a reason to replan; it is never silently treated as a complete change analysis. Chapter 06 adds a human review gate.
